# Skala water flows on Colab (GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xryptowolfe/skala/blob/main/flows/water/water_flows_colab.ipynb)

1. **Runtime → Change runtime type → GPU** (T4 is enough for water).
2. Run the cells in order. If Colab asks to restart after the install, restart and continue from the cell after it.

Same commands as on a laptop, with `--device cuda`. See `flows/water/README.md` for all options.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

Install. The cell reads the driver's CUDA version and installs the matching `skala-cuda12x` or `skala-cuda13x` (GPU4PySCF, CuPy and PyTorch for that CUDA). The fork's own `skala/` is then installed on top without touching those dependencies. Set `BRANCH` to try another branch of the fork.

In [ ]:
BRANCH = "main"
!git clone -q -b {BRANCH} https://github.com/xryptowolfe/skala.git /content/skala
%cd /content/skala
import re, subprocess
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
cuda_major = int(re.search(r"CUDA Version: (\d+)", smi).group(1))
pkg = "skala-cuda13x" if cuda_major >= 13 else "skala-cuda12x"
print(f"driver supports CUDA {cuda_major} -> installing {pkg}")
!pip install -q {pkg} matplotlib
!pip install -q --no-deps -e ./skala
import torch, skala
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Optimize, frequencies, thermochemistry

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m flows.water sp opt freq thermo --device cuda 2>&1 | grep -v '^Warning'

## Energy scans (H-O-H bend and O-H stretch)

In [ ]:
import glob
opt = sorted(glob.glob('runs/*opt*/optimized.xyz'))[-1]
!CUDA_VISIBLE_DEVICES=0 python -m flows.water scan --geometry {opt} --device cuda 2>&1 | grep -v '^Warning' | tail -3
!CUDA_VISIBLE_DEVICES=0 python -m flows.water scan --scan-coord stretch --geometry {opt} --device cuda 2>&1 | grep -v '^Warning' | tail -3
from IPython.display import Image, display
for png in sorted(glob.glob('runs/*scan*/scan_*.png'))[-2:]:
    display(Image(png))

## Download all results

In [ ]:
!zip -qr water_runs.zip runs
from google.colab import files
files.download('water_runs.zip')